In [9]:
import os
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

# 1. Laeme Supabase ühenduse andmed .env failist
load_dotenv()

supabase_url = os.getenv('SUPABASE_URL')
supabase_key = os.getenv('SUPABASE_KEY')

supabase = create_client(supabase_url, supabase_key)


# 2. Abifunktsioon: laeb Supabase'ist kõik tabeli read lehekülgede kaupa
def get_data(table_name):
  data = []
  page_size = 1000
  page = 0
  while True:
    response = (
        supabase.table(table_name)
        .select('*')
        .range(page * page_size, (page + 1) * page_size - 1)
        .execute()
    )
    data.extend(response.data)
    if len(response.data) < page_size:
      break
    page += 1
  return pd.DataFrame(data)


# 3. Laeme andmed kõigist kolmest tabelist
sales_df = get_data('sales')
customers_df = get_data('customers')
products_df = get_data('products')

# Kontrollime laetud ridade arvu ja veergude olemasolu
print(f'sales_df: {len(sales_df)} rida')
print(f'customers_df: {len(customers_df)} rida')
print(f'products_df: {len(products_df)} rida\n')

# 4. Ühendame tabelid (Merge / JOIN)
# Müügid + Kliendid ('customer_id' alusel)
sales_customers_df = pd.merge(
    sales_df, customers_df, on='customer_id', how='left'
)

# Müügid + Tooted ('product_id' alusel)
sales_products_df = pd.merge(
    sales_df, products_df, on='product_id', how='left'
)

# 5. Analüüs 1: TOP 5 enim käivet toonud toodet
top_products = (
    sales_products_df.groupby('product_name')['total_price'].sum().nlargest(5)
)

print('=== TOP 5 TOODET KÄIBE JÄRGI ===')
print(top_products)
print('\n')

# 6. Analüüs 2: Suurima ostusummaga klient (Top customer)
sales_customers_df['full_name'] = (
    sales_customers_df['first_name'] + ' ' + sales_customers_df['last_name']
)

top_customer = (
    sales_customers_df.groupby('full_name')['total_price'].sum().nlargest(1)
)


sales_df: 10118 rida
customers_df: 3150 rida
products_df: 362 rida

=== TOP 5 TOODET KÄIBE JÄRGI ===
product_name
Õhuline sünteetiline sporditossud    27347.04
Trendikas goretex oxfordid           23376.15
Praktiline viskoosne jakk            22188.80
Praktiline džersii seelik            22039.98
Boheemlaslik puuvillane tuulejope    21309.96
Name: total_price, dtype: float64


